In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Vertex AI Memory Bank 入门

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/memory_bank/get_started_with_memory_bank.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fagents%2Fagent_engine%2Fmemory_bank%2Fget_started_with_memory_bank.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/agents/agent_engine/memory_bank/get_started_with_memory_bank.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/memory_bank/get_started_with_memory_bank.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>


| Authors |
| --- |
| [Kimberly Milam](https://github.com/klmilam) |
| [Ivan Nardini](https://github.com/inardini) |

## 概述

本 Notebook 是掌握 **Vertex AI Memory Bank** 的实践指南，该服务用于构建有状态、上下文感知的对话式 AI Agent。您将学习如何赋予 Agent 持久的长期记忆，使其能够跨多个会话回忆宾客偏好和过往互动，从而提供真正个性化的酒店体验。我们将在一个实际的酒店场景中应用这些概念：构建一个复杂的酒店礼宾助手。

完成本教程后，您不仅能理解 Memory Bank 的核心概念，还能了解如何将其应用于构建一个能记住宾客偏好、饮食限制、房间偏好的助手，并在对话间保持上下文，从而提供卓越的个性化服务。

以下是我们将采取的步骤概览：

* **初始设置**：我们将从基础开始，配置一个新的 Memory Bank 实例，并学习如何创建宾客会话来存储和检索对话历史。
* **基本记忆操作**：我们将探索如何从对话中生成记忆，并通过检索了解系统对每位宾客的记忆内容。
* **实际应用**：我们将了解如何在宾客再次入住酒店时利用这些记忆来个性化宾客互动。
* **资源管理**：最后，我们将通过适当清理资源来处理基本的运营事项。


## 开始



### 安装 Vertex AI SDK 和所需软件包

首先，让我们安装 Vertex AI SDK。

**注意**：这将安装 SDK。Colab 可能会在安装后提示您重启运行时。这是预期行为。


In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform

### 验证您的 Notebook 环境

如果您在 **Google Colab** 中运行此 Notebook，请运行下面的单元格以验证您的账户。


In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### 设置 Google Cloud 项目信息

要开始使用 Vertex AI，您必须拥有一个现有的 Google Cloud 项目，并[启用 Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)。

了解更多关于[设置项目和开发环境](https://cloud.google.com/vertex-ai/docs/start/cloud-environment)的信息。


In [ ]:
import os

import vertexai

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

# 初始化 Vertex AI 客户端
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

print("✅ Vertex AI 客户端已初始化！")
print(f"   项目：{PROJECT_ID}")
print(f"   位置：{LOCATION}")


### 导入库

我们正在导入标准 Python 库以及来自 Vertex AI SDK 的几种基于类的类型。

为了使代码更易读，我们为这些较长的类名创建了较短的别名。这是一种常见的 Python 实践，有助于保持代码简洁，同时不失使用类型化类的优势。


In [ ]:
import datetime
import os
import uuid
import warnings

warnings.filterwarnings("ignore")

# 导入 Memory Bank 的基于类的类型
from vertexai import types

# 基本配置类型
MemoryBankConfig = types.ReasoningEngineContextSpecMemoryBankConfig
SimilaritySearchConfig = (
    types.ReasoningEngineContextSpecMemoryBankConfigSimilaritySearchConfig
)
GenerationConfig = types.ReasoningEngineContextSpecMemoryBankConfigGenerationConfig

# 高级配置类型
TtlConfig = types.ReasoningEngineContextSpecMemoryBankConfigTtlConfig
GranularTtlConfig = (
    types.ReasoningEngineContextSpecMemoryBankConfigTtlConfigGranularTtlConfig
)
CustomizationConfig = types.MemoryBankCustomizationConfig
MemoryTopic = types.MemoryBankCustomizationConfigMemoryTopic
ManagedMemoryTopic = types.MemoryBankCustomizationConfigMemoryTopicManagedMemoryTopic
CustomMemoryTopic = types.MemoryBankCustomizationConfigMemoryTopicCustomMemoryTopic
GenerateMemoriesExample = types.MemoryBankCustomizationConfigGenerateMemoriesExample
ConversationSource = (
    types.MemoryBankCustomizationConfigGenerateMemoriesExampleConversationSource
)
ConversationSourceEvent = (
    types.MemoryBankCustomizationConfigGenerateMemoriesExampleConversationSourceEvent
)
ExampleGeneratedMemory = (
    types.MemoryBankCustomizationConfigGenerateMemoriesExampleGeneratedMemory
)
ManagedTopicEnum = types.ManagedTopicEnum

print("✅ 库导入成功！")


### 定义显示记忆的辅助函数

此辅助函数在整个教程中显示生成的记忆时提供一致的格式。


In [ ]:
def display_generated_memories(operation, client, title="Generated Memories"):
    """以一致的格式显示来自生成操作的记忆。

    Args:
        operation: client.agent_engines.memories.generate() 的返回结果
        client: Vertex AI 客户端实例
        title: 显示在记忆上方的标题
    """
    if operation.response and operation.response.generated_memories:
        print(f"\n✅ {title}: {len(operation.response.generated_memories)}\n")

        for i, gen_memory in enumerate(operation.response.generated_memories, 1):
            if gen_memory.action != "DELETED" and gen_memory.memory:
                try:
                    full_memory = client.agent_engines.memories.get(
                        name=gen_memory.memory.name
                    )
                    action_icon = "🆕" if gen_memory.action == "CREATED" else "��"
                    print(f"   {action_icon} {i}. {full_memory.fact}")
                except Exception as e:
                    print(f"   ⚠️ 无法检索记忆：{e}")
    else:
        print(f"\n📭 未找到{title.lower()}")


print("✅ 辅助函数已定义！")


## 创建您的酒店礼宾 Memory Bank

现在让我们通过创建具有 Memory Bank 功能的 Agent Engine 来为酒店礼宾助手奠定基础。


### 创建带有 Memory Bank 配置的 Agent Engine

AgentEngine 资源充当 Memory Bank 实例的顶级容器。要创建一个，我们需要提供配置。

其中，MemoryBankConfig 有两个关键部分：

1. **`similarity_search_config`**：这指定了用于相似性搜索的**嵌入模型**。当宾客询问"我上次要求了什么？"时，此模型有助于找到最相关的记忆。我们使用 `text-embedding-005`，它非常适合英语对话。如果您预期有多语言宾客，请考虑使用 `text-multilingual-embedding-002`。

2. **`generation_config`**：这定义了将从对话中提取和整合记忆的 **LLM**。默认值 `gemini-2.5-flash` 是一个快速且功能强大的模型，非常适合此任务。它读取对话并智能地提取关键事实，如饮食偏好或房间温度偏好。


In [ ]:
print("🧠 正在为酒店礼宾创建 Memory Bank 配置...\n")

basic_memory_config = MemoryBankConfig(
    # 用于相似性搜索的嵌入模型
    similarity_search_config=SimilaritySearchConfig(
        embedding_model=f"projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/models/text-embedding-005"
    ),
    # 用于从对话中提取记忆的 LLM
    generation_config=GenerationConfig(
        model=f"projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/models/gemini-2.5-flash"
    ),
)

print("✅ Memory Bank 配置已创建！")
print("   嵌入模型：text-embedding-005")
print("   生成模型：gemini-2.5-flash")


现在，我们创建 AgentEngine 资源。默认情况下，创建 Agent Engine 时会启用 Memory Bank。此调用会配置必要的后端基础设施来存储和检索宾客记忆。


In [ ]:
print("\n🛠️ 正在创建带有 Memory Bank 的 Agent Engine...\n")
print("⏳ 这将为宾客记忆存储配置后端基础设施...")

agent_engine = client.agent_engines.create(
    config={"context_spec": {"memory_bank_config": basic_memory_config}}
)

agent_engine_name = agent_engine.api_resource.name

print("\n✅ Agent Engine 创建成功！")
print(f"   资源名称：{agent_engine_name}")


## 存储宾客对话

现在让我们模拟一次真实的酒店入住对话，看看 Memory Bank 如何提取和存储重要的宾客信息。


### 为宾客创建会话

**Session（会话）** 是宾客与礼宾 Agent 之间单次交互的时间顺序日志，是生成记忆的原始材料。

每个会话都与一个 `user_id`（在本例中为宾客的标识符）相关联。这使 Agent 能够跨不同住宿和交互回忆特定宾客的信息。可以将其视为酒店系统中的宾客档案 ID。

**重要提示**：使用 Vertex AI Agent Engine Session 并非唯一支持的选项。如果您使用不同的会话存储系统，也可以[直接以 JSON 格式提供源对话](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/generate-memories#json-format)。


In [ ]:
print("💬 正在为宾客入住创建会话...\n")

# 生成唯一的宾客标识符
guest_id = "guest_emma_" + str(uuid.uuid4())[:4]

# 为此宾客创建会话
session = client.agent_engines.sessions.create(
    name=agent_engine_name,
    user_id=guest_id,
    config={"display_name": f"Check-in conversation for {guest_id}"},
)

session_name = session.response.name

print("✅ 会话创建成功！")
print(f"   会话：{session_name}")
print(f"   宾客 ID：{guest_id}")


### 定义宾客对话

这是我们将使用的原始对话数据。它是一个简单的字典列表，每个字典代表宾客与酒店礼宾之间对话中的一个轮次。

此对话包含有关宾客偏好的宝贵信息，我们希望将其记住以便未来住宿使用。请注意宾客如何提及饮食限制、房间偏好和特殊请求。

在阅读此对话时，想想一个好的礼宾会记住什么：
- 宾客的姓名（Emma）
- 饮食限制（素食、乳糖不耐受）
- 房间偏好（高楼层、远离电梯、安静）
- 特殊请求（额外枕头、白噪音机）
- 温度偏好（凉爽的房间，68°F）


In [ ]:
# 一次真实的酒店入住对话
guest_conversation = [
    {
        "role": "user",
        "message": "Hi! I'm Emma Chen. I have a reservation for a deluxe room for three nights.",
    },
    {
        "role": "model",
        "message": "Welcome to our hotel, Ms. Chen! I have your reservation right here. Let me get you checked in.",
    },
    {
        "role": "user",
        "message": "Great! Just so you know, I'm vegetarian and lactose intolerant, so please note that for room service and the breakfast buffet.",
    },
    {
        "role": "model",
        "message": "Absolutely, I've noted your dietary preferences. Our kitchen will be informed, and we have excellent vegetarian options.",
    },
    {
        "role": "user",
        "message": "Thank you! Also, I prefer a high floor room, away from the elevator. I'm a light sleeper, so somewhere quiet would be ideal.",
    },
    {
        "role": "model",
        "message": "Of course! I'll assign you a corner room on the 12th floor. It's one of our quietest locations.",
    },
    {
        "role": "user",
        "message": "Perfect! One more thing - can I get extra pillows and a white noise machine? And I prefer the room temperature around 68°F.",
    },
    {
        "role": "model",
        "message": "Certainly! I'll have housekeeping bring extra pillows and a white noise machine to your room, and I'll set the thermostat to 68°F before your arrival.",
    },
]

print("✅ Guest conversation defined!")
print(f"   {len(guest_conversation)} conversation turns")


### 将对话添加到会话

在这里，我们循环遍历对话，并将每个轮次作为事件追加到我们之前创建的会话中。这会持久保存对话历史，使其可用于记忆生成。

Memory Bank 需要在分析和提取记忆之前，将完整的对话存储在会话中。每个轮次都有时间戳，并归属于宾客（user）或礼宾（model）。


In [ ]:
print("⬆️ 正在将对话添加到会话...\n")

invocation_id = 0

for turn in guest_conversation:
    client.agent_engines.sessions.events.append(
        name=session_name,
        author=guest_id,  # Sessions 所需
        invocation_id=str(invocation_id),  # Sessions 所需
        timestamp=datetime.datetime.now(
            tz=datetime.timezone.utc
        ),  # Sessions 所需
        config={
            "content": {"role": turn["role"], "parts": [{"text": turn["message"]}]}
        },
    )

    invocation_id += 1
    icon = "👤" if turn["role"] == "user" else "🤖"
    print(f"{icon} {turn['message']}")

print("\n✅ 对话已成功添加到会话！")


## 从对话中生成记忆

现在让我们看看 AI 从这次对话中自动记住了什么。


### 提取并整合宾客记忆

这是记忆生成的核心。generate 方法启动一个执行两个主要步骤的操作。

Memory Bank 的工作原理如下：

1. **提取**：生成模型（gemini-2.5-flash）读取对话并提取关键事实。使用默认配置，它会查找与预定义**托管主题**匹配的信息，例如：
   - `USER_PERSONAL_INFO`：个人详细信息，如宾客的姓名
   - `USER_PREFERENCES`：关于房间、设施、温度等的偏好声明
   - `EXPLICIT_INSTRUCTIONS`：宾客提出的具体请求

2. **整合**：Memory Bank 智能地将新事实与现有记忆（如果宾客之前住过）合并，避免重复并解决矛盾。例如，如果宾客之前偏好 70°F，但现在说 68°F，系统会更新记忆。


**注意**：`wait_for_completion=True` 标志使这成为一个阻塞调用，这对本教程很有用。在生产环境中，您通常会将其设置为 `False` 以在后台运行，稍后轮询结果。


In [ ]:
print("🧠 正在分析对话并提取记忆...\n")
print(
    "⏳ AI 正在读取对话并识别要记忆的重要事实..."
)

# 从会话生成记忆
operation = client.agent_engines.memories.generate(
    name=agent_engine_name,
    vertex_session_source={"session": session_name},
    config={"wait_for_completion": True},
)

print("\n✅ 记忆提取完成！")


### 显示提取的记忆

让我们看看 AI 从 Emma 的入住对话中记住了什么。

Memory Bank 将提取有关以下内容的结构化事实：
- 宾客身份和个人信息
- 饮食偏好和限制
- 房间位置偏好
- 温度偏好
- 特殊设施请求

每条记忆都是一个简洁的事实性陈述，可用于个性化未来的互动。


In [ ]:
# 使用辅助函数显示生成的记忆
display_generated_memories(operation, client, "Guest Preferences Extracted")


## 检索和使用宾客记忆

现在让我们了解如何检索这些记忆并用它们来个性化宾客体验。


### 检索宾客的所有记忆

让我们检索刚刚为宾客创建的所有记忆。最简单的方法是基于范围的检索。

"scope（范围）"是一组键值对，定义了一组记忆。通过提供 `{"user_id": guest_id}`，我们请求属于这位特定宾客的所有记忆。这类似于查询数据库："给我所有 user_id = 'guest_emma_1234' 的记录"

当宾客返回或致电前台时，您希望快速访问您了解的所有关于其偏好的信息。


In [ ]:
print(f"📚 正在检索宾客 {guest_id} 的所有记忆...\n")

# 简单检索——获取此宾客的所有记忆
results = client.agent_engines.memories.retrieve(
    name=agent_engine_name, scope={"user_id": guest_id}
)

all_memories = list(results)

print(f"✅ 找到该宾客的 {len(all_memories)} 条记忆：\n")

for i, retrieved_memory in enumerate(all_memories, 1):
    print(f"   {i}. {retrieved_memory.memory.fact}")


## 综合运用

使用 Vertex AI Memory Bank，当 Emma 再次入住时，我们可以立即调取她的偏好，提供卓越的个性化服务。因此，无需让 Emma 重复所有偏好，我们可以主动按照她喜欢的方式布置好房间。


In [ ]:
print("=" * 80)
print("场景：宾客再次入住")
print("=" * 80)
print("\n📅 两周后，Emma 再次入住酒店...\n")

# Simulate a return visit conversation
print("👤 Hi! I'm Emma Chen, checking in again.")
print("�� Welcome back, Ms. Chen! Let me pull up your preferences...\n")

# Retrieve Emma's memories
print("🔍 正在从上次住宿中检索宾客偏好...")
results = client.agent_engines.memories.retrieve(
    name=agent_engine_name, scope={"user_id": guest_id}
)

memories = list(results)

print(f"✅ 找到 {len(memories)} 条已存档偏好\n")

# Display how we'll use these memories
print("🤖 Perfect! Based on your previous stay, I have:")
print()

for memory in memories:
    fact = memory.memory.fact

    # Provide contextual responses based on memory content
    if "vegetarian" in fact.lower() or "lactose" in fact.lower():
        print(f"   🥗 Dietary: {fact}")
        print("      → I've already notified the kitchen and breakfast staff")
    elif (
        "floor" in fact.lower() or "quiet" in fact.lower() or "elevator" in fact.lower()
    ):
        print(f"   🏨 Room: {fact}")
        print("      → I've assigned you the same corner room on floor 12")
    elif "temperature" in fact.lower() or "68" in fact:
        print(f"   🌡️  Climate: {fact}")
        print("      → Thermostat pre-set before your arrival")
    elif "pillow" in fact.lower() or "white noise" in fact.lower():
        print(f"   🛏️  Amenities: {fact}")
        print("      → Already placed in your room")
    else:
        print(f"   ℹ️  {fact}")

    print()


## 高级检索：相似性搜索

在前面的示例中，我们使用基于范围的检索获取了宾客的**所有**记忆。但如果您只想获取特定问题中最**相关**的记忆怎么办？这正是**相似性搜索**的优势所在。


### 理解差异：范围 vs. 相似性

想象一下，Emma 已在您的酒店住过 10 次，存储了 50 多条记忆。当前台需要回答"Emma 的饮食限制是什么？"时，他们不需要所有 50 条记忆——只需要 2-3 条最相关的食物偏好信息。

这就是记忆相似性搜索的意义所在。总结一下：

- **基于范围的检索**：返回匹配范围的所有记忆（例如，user_id = "guest_emma_1234" 的所有记忆）
- **相似性搜索**：使用语义相似性为特定查询返回最相关的 K 条记忆

让我们通过为 Emma 添加更多对话历史来实际演示。


#### 添加更多宾客互动

Emma 致电前台，询问有关水疗服务和早餐选项的问题。让我们记录这次对话。


In [ ]:
print("📞 Emma 致电前台询问一些问题...\n")

# 额外的对话轮次
additional_conversation = [
    {
        "role": "user",
        "message": "Hi, I'd like to book a spa treatment tomorrow. Do you have deep tissue massage? I prefer morning appointments, around 9 AM.",
    },
    {
        "role": "model",
        "message": "Absolutely! I can book you a deep tissue massage at 9 AM tomorrow. I've noted your preference for morning spa appointments.",
    },
    {
        "role": "user",
        "message": "Great! Also, what vegetarian options do you have for breakfast? I need dairy-free choices because of my lactose intolerance.",
    },
    {
        "role": "model",
        "message": "We have several vegetarian and dairy-free options: oat milk smoothie bowls, avocado toast, fresh fruit platters, and coconut yogurt parfaits.",
    },
]

# 将这些轮次添加到同一会话
invocation_id_counter = len(guest_conversation)

for turn in additional_conversation:
    client.agent_engines.sessions.events.append(
        name=session_name,
        author=guest_id,
        invocation_id=str(invocation_id_counter),
        timestamp=datetime.datetime.now(tz=datetime.timezone.utc),
        config={
            "content": {"role": turn["role"], "parts": [{"text": turn["message"]}]}
        },
    )

    invocation_id_counter += 1
    icon = "👤" if turn["role"] == "user" else "🤖"
    print(f"{icon} {turn['message']}")

print("\n✅ 额外对话已添加到会话！")


#### 从扩展对话中生成新记忆

现在让我们从这个扩展对话中生成记忆。Memory Bank 将分析整个会话并提取新事实，同时将其与现有记忆整合。

这演示了 Memory Bank 的整合能力——它不会创建重复记忆。相反，它会将新信息与已知的 Emma 信息合并。


In [ ]:
print("\n🧠 正在从扩展对话生成记忆...\n")
print("⏳ 正在分析完整对话历史并整合记忆...")

# 生成额外记忆
operation = client.agent_engines.memories.generate(
    name=agent_engine_name,
    vertex_session_source={"session": session_name},
    config={"wait_for_completion": True},
)

print("\n✅ 记忆生成完成！")


In [ ]:
# 显示新生成的记忆
display_generated_memories(operation, client, "New Memories Generated")


#### 比较基于范围 vs. 基于相似性的检索

现在让我们并排比较两种检索方法，看看差异。


In [ ]:
print("比较：基于范围 vs. 基于相似性的检索")

# 方法 1：基于范围的检索（返回所有记忆）
print("\n📊 方法 1：基于范围的检索")
print("查询：获取 Emma 的所有记忆")
print("-" * 80)

all_results = client.agent_engines.memories.retrieve(
    name=agent_engine_name, scope={"user_id": guest_id}
)

all_memories = list(all_results)
print(f"\n✅ 找到 {len(all_memories)} 条总记忆：\n")

for i, mem in enumerate(all_memories, 1):
    print(f"   {i}. {mem.memory.fact}")

# 方法 2：相似性搜索（仅返回相关记忆）
print("\n🔍 方法 2：相似性搜索")
print("查询：Emma 的饮食限制是什么？")
print("-" * 80)

search_results = client.agent_engines.memories.retrieve(
    name=agent_engine_name,
    scope={"user_id": guest_id},
    similarity_search_params={
        "search_query": "What are Emma's dietary restrictions?",
        "top_k": 3,  # 获取最相关的 3 条记忆
    },
)

relevant_memories = list(search_results)
print(f"\n✅ 找到 {len(relevant_memories)} 条相关记忆：\n")

for i, mem in enumerate(relevant_memories, 1):
    distance = mem.distance if hasattr(mem, "distance") else "N/A"
    print(f"   {i}. {mem.memory.fact}")
    print(f"      → 相关性距离：{distance}（越低 = 越相关）")


### 使用相似性搜索回答特定问题

让我们用几个具体的前台问题来测试相似性搜索。

在实际场景中，酒店员工需要快速、有针对性的答案。相似性搜索通过仅检索与每个特定问题相关的内容，使这一过程更加高效。

**理解相似性搜索参数**

| 参数 | 描述 | 示例值 |
|-----------|-------------|---------------|
| `search_query` | 自然语言问题或主题 | "宾客的房间偏好是什么？" |
| `top_k` | 最大返回结果数量 | 3（返回最相关的 3 条） |
| `distance` | 查询与记忆嵌入之间的距离 | 自动计算（越低 = 越相似） |


In [ ]:
print("\n🏨 使用常见前台问题测试相似性搜索\n")

# 使用各种具体查询进行测试
search_queries = [
    "What are Emma's room location preferences?",
    "What are Emma's dietary restrictions?",
    "What spa services does Emma prefer?",
    "What temperature does Emma like in her room?",
]

for query in search_queries:
    print(f'\n❓ Query: "{query}"')
    print("-" * 80)

    # 执行相似性搜索
    results = client.agent_engines.memories.retrieve(
        name=agent_engine_name,
        scope={"user_id": guest_id},
        similarity_search_params={
            "search_query": query,
            "top_k": 2,  # 获取最相关的 2 条记忆
        },
    )

    memories = list(results)

    if memories:
        print("\n🎯 最相关记忆：")
        for i, mem in enumerate(memories, 1):
            distance = mem.distance if hasattr(mem, "distance") else "N/A"
            print(f"   {i}. {mem.memory.fact}")
            print(
                f"      (Distance: {distance:.4f})"
                if isinstance(distance, (int, float))
                else f"      (Distance: {distance})"
            )
    else:
        print("   未找到相关记忆")

    print()

print("✅ 相似性搜索可实现快速、有针对性的信息检索！")


### 最佳实践：何时使用每种检索方法

**使用基于范围的检索的情况**：
- 您需要宾客的完整档案
- 您在仪表板中显示所有偏好
- 记忆数量较少
- 您希望确保不遗漏任何内容

**使用相似性搜索的情况**：
- 您在回答特定问题
- 宾客有许多记忆
- 您需要快速、有针对性的响应
- 您正在构建需要上下文感知响应的对话式 Agent


## 清理

为避免因本教程中使用的资源而对您的 Google Cloud 账户产生费用，请删除我们创建的 Agent Engine。


In [ ]:
print("🧹 正在清理资源...\n")

delete_agent_engine = True

if delete_agent_engine:
    # 删除 Agent Engine 及其所有记忆
    client.agent_engines.delete(name=agent_engine_name, force=True)

    print("✅ Agent Engine 已成功删除！")
    print("   所有宾客记忆已被移除")
else:
    print("⏭️  跳过清理——Agent Engine 将保持活跃")
    print(f"   资源：{agent_engine_name}")
    print("\n⚠️  请记得稍后删除此资源以避免产生费用")


## 恭喜！

您已完成"Vertex AI Memory Bank 入门"教程！

您现在拥有构建上下文感知、个性化 AI Agent 的基础知识，这些 Agent 能够跨会话记住用户偏好。您在此构建的酒店礼宾场景可以适用于无数个性化和记忆至关重要的用例。

**下一步？**
- 在我们的中级教程中探索 Memory Bank 的高级功能
- 查看 [Memory Bank 文档](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank)
- 加入 [Google Cloud AI 社区](https://discuss.google.dev/c/google-cloud/14) 分享您的项目
